# Hybrid Reranker Demo
This notebook demonstrates the high-precision retrieval pipeline:
1. **Stage 1 (Recall):** Reciprocal Rank Fusion (RRF) across Vector (Qdrant) and Sparse (BM25) search.
2. **Stage 2 (Precision):** Cross-Encoder scoring (`ms-marco-MiniLM-L-6-v2`) of the candidate pool.
3. **Trust:** Full citation metadata including PDF bounding boxes and Merkle hash verification.

In [ ]:
import os
import sys
import asyncio

# Add 'src' to path
if "./src" not in sys.path:
    sys.path.insert(0, "./src")

from ingestion_pipeline.ingestion_pipeline import AsyncMerkleQdrantIngestor
from reranker_pipeline.reranker_pipeline import HybridReranker, format_citation

# 1. Initialize Ingestor (Connects to Qdrant)
ingestor = AsyncMerkleQdrantIngestor(qdrant_url="http://localhost:6333")
await ingestor.setup()

# 2. Initialize Reranker
# alpha=0.7 balances Cross-Encoder (accuracy) vs RRF (consensus)
reranker = HybridReranker(ingestor=ingestor, alpha=0.7)

# 3. Run Hybrid Reranking
query = "vulnerability in older age"
print(f"Querying: {query}\n" + "="*50)

results = await reranker.rerank(
    query=query,
    retrieval_top_k=20,  # Grab 20 candidates per leg
    rerank_top_n=3,      # Return top 3 precision-scored results
)

# 4. Display Results with full Citations
if not results:
    print("No results found in Qdrant. Make sure documents are ingested first!")
else:
    for i, res in enumerate(results, start=1):
        print(format_citation(res, index=i))
        print(f"Content snippet: {res.content[:300]}...")
        print("-" * 40)

# Check cache usage (Cross-Encoder scores are cached in-memory)
print("\nReranker Cache Stats:", reranker.cache_stats())